# ALS baseline for popularity calibration

**Owner:** Daryl Shy  
**Due:** October 12, 2026  
**Scope:** Spark/Scala environment, ALS baseline, and baseline recommendation outputs.

This notebook will supply relevance scores and candidate movies for the study's later calibration methods. This first version verifies the environment only. The ALS model and Top-N outputs have not been implemented.

Run the implemented cells from top to bottom. Each collaborator needs their own compatible local kernel; GitHub stores this notebook, not its running session.


## 1. Verify the local environment

Before we load ratings or train anything, check that the notebook can load Spark, start a local session, and complete a tiny operation. This isolates environment problems from later data or model problems.

The `$ivy` imports load the specified Spark libraries into this Scala notebook. `spark-sql` provides structured data processing; `spark-mllib` supplies the machine-learning library we will use later. No ALS model is trained here.


In [1]:
import $ivy.`org.apache.spark::spark-sql:3.5.9`
import $ivy.`org.apache.spark::spark-mllib:3.5.9`

import org.apache.spark.sql.SparkSession


import $ivy.$
import $ivy.$
import org.apache.spark.sql.SparkSession

### Start Spark and check a small result

`val spark` gives the session a name we can reuse. `.builder()` starts its configuration, and each following method adds one setting. `local[2]` runs Spark locally with two worker threads. The loopback settings keep this local notebook's driver reachable on this machine. `.getOrCreate()` returns an existing session or creates one.

`spark.range(3)` creates three synthetic rows; `.count()` runs a calculation and returns their count. This is an environment check, not MovieLens data. The assertions stop this check if the runtime versions or count differ from the expected setup.


In [2]:
val spark = SparkSession.builder()
  .appName("CS 7265 - ALS Baseline Environment")
  .master("local[2]")
  .config("spark.driver.host", "127.0.0.1")
  .config("spark.driver.bindAddress", "127.0.0.1")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
val scalaVersion = scala.util.Properties.versionNumberString
val javaVersion = System.getProperty("java.version")
val checkCount = spark.range(3).count()

assert(spark.version == "3.5.9")
assert(scalaVersion == "2.12.18")
assert(System.getProperty("java.specification.version") == "17")
assert(checkCount == 3L)

println(s"Spark: ${spark.version}")
println(s"Scala: $scalaVersion")
println(s"Java: $javaVersion")
println(s"Environment check passed: $checkCount synthetic rows.")


Spark: 3.5.9


Scala: 2.12.18


Java: 17.0.16


Environment check passed: 3 synthetic rows.


spark: SparkSession = org.apache.spark.sql.SparkSession@59a60dfd
scalaVersion: String = "2.12.18"
javaVersion: String = "17.0.16"
checkCount: Long = 3L

### Release the check session

The setup check is complete. `spark.stop()` releases its resources. Run the session cell again when starting new work that needs Spark.


In [3]:
spark.stop()
println("Environment-check session stopped.")


Environment-check session stopped.


**MarginNote:** The notebook holds our explanation, code, and saved results. The Scala kernel executes the code; Spark performs the data processing. GitHub versions the saved files for review.


## 2. Implement the ALS baseline

We will first explain what the observed user/movie ratings represent and how ALS estimates missing ratings. Implementation will proceed one block at a time.


## 3. Generate baseline recommendations

The proposal calls for scoring candidate movies absent from each user's training history, retaining the top 100, scaling their scores to 0–1, and taking the first 10 for the unadjusted baseline. The agreed protocol and training input must be established before producing final baseline outputs.
